<a href="https://colab.research.google.com/github/aaneesa/SectionD_Gr18_Spotify-User-Behavior-And-Pattern/blob/main/notebooks/02_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import pandas as pd
import numpy as np


In [12]:
df = pd.read_csv('spotify_user_behavior_realistic_50000_rows.csv')


In [ ]:
df.head()



,user_id,country,age,signup_date,subscription_type,subscription_status,months_inactive,inactive_3_months_flag,ad_interaction,ad_conversion_to_subscription,music_suggestion_rating_1_to_5,avg_listening_hours_per_week,favorite_genre,most_liked_feature,desired_future_feature,primary_device,playlists_created,avg_skips_per_day
0,1,France,25,2021-08-19,Premium Duo,Active,0,0,No,No,4,10.13,Bollywood,Radio,Concert Alerts,Tablet,7,8
1,2,Indonesia,20,2022-06-06,Premium Family,Active,0,0,Yes,No,5,11.63,Latin,Podcasts,Lyrics Translation,Mobile,7,6
2,3,Italy,53,2024-01-04,Premium Individual,Active,0,0,Yes,Yes,3,9.50,Bollywood,Lyrics,Better AI Recommendations,Desktop,6,5
3,4,Italy,48,2018-08-26,Premium Individual,Active,1,0,No,No,4,13.16,Electronic,Playlists,Social Listening,Smart Speaker,11,8
4,5,Australia,18,2020-05-29,Free,Active,0,0,No,No,4,12.70,Indie,Daily Mix,Lyrics Translation,Tablet,10,11


In [13]:
df.info()        # structure of data


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   user_id                         50000 non-null  int64  
 1   country                         50000 non-null  object 
 2   age                             50000 non-null  int64  
 3   signup_date                     50000 non-null  object 
 4   subscription_type               50000 non-null  object 
 5   subscription_status             50000 non-null  object 
 6   months_inactive                 50000 non-null  int64  
 7   inactive_3_months_flag          50000 non-null  int64  
 8   ad_interaction                  50000 non-null  object 
 9   ad_conversion_to_subscription   50000 non-null  object 
 10  music_suggestion_rating_1_to_5  50000 non-null  int64  
 11  avg_listening_hours_per_week    50000 non-null  float64
 12  favorite_genre                  

In [15]:
df.describe()    # statistical summary


,user_id,age,months_inactive,inactive_3_months_flag,music_suggestion_rating_1_to_5,avg_listening_hours_per_week,playlists_created,avg_skips_per_day
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,25000.500000,38.010280,1.533020,0.222460,3.644100,9.988986,8.002680,10.025920
std,14433.901067,12.984989,1.952082,0.415903,1.114424,3.968927,2.831571,3.165579
min,1.000000,16.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000
25%,12500.750000,27.000000,0.000000,0.000000,3.000000,7.280000,6.000000,8.000000
50%,25000.500000,38.000000,1.000000,0.000000,4.000000,9.980000,8.000000,10.000000
75%,37500.250000,49.000000,2.000000,0.000000,5.000000,12.680000,10.000000,12.000000
max,50000.000000,60.000000,18.000000,1.000000,5.000000,26.250000,23.000000,25.000000


In [16]:
df.shape         # rows, columns

(50000, 18)

In [17]:
df.isnull().sum()


,0
user_id,0
country,0
age,0
signup_date,0
subscription_type,0
subscription_status,0
months_inactive,0
inactive_3_months_flag,0
ad_interaction,0
ad_conversion_to_subscription,0


there are no missing values in any of our columns so fillna will not take that.

In [18]:
df = df.drop_duplicates()


In [19]:
df.duplicated().sum()


np.int64(0)

In [20]:
df.dtypes


,0
user_id,int64
country,object
age,int64
signup_date,object
subscription_type,object
subscription_status,object
months_inactive,int64
inactive_3_months_flag,int64
ad_interaction,object
ad_conversion_to_subscription,object


In [21]:
# Convert to datetime
df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')

# Drop invalid dates
df = df.dropna(subset=['signup_date'])

# Create user age (days)
today = pd.Timestamp.today()
df['user_age_days'] = (today - df['signup_date']).dt.days


IQR Outlier Detection (Listening Hours)

In [22]:
Q1 = df['avg_listening_hours_per_week'].quantile(0.25)
Q3 = df['avg_listening_hours_per_week'].quantile(0.75)
IQR = Q3 - Q1


In [23]:
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR


Flag outliers

In [24]:
df['is_outlier'] = (
    (df['avg_listening_hours_per_week'] < lower_bound) |
    (df['avg_listening_hours_per_week'] > upper_bound)
)


Remove outliers

In [25]:
df = df[
    (df['avg_listening_hours_per_week'] >= lower_bound) &
    (df['avg_listening_hours_per_week'] <= upper_bound)
]


Age Segmentation

In [26]:
bins = [0, 18, 24, 34, 44, 54, 100]
labels = ['<18', '18-24', '25-34', '35-44', '45-54', '55+']

df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels)


Engagement Segmentation

Active–Inactive Paradox Fix

In [29]:
df.rename(columns={'inactive_3_months_flag': 'is_churned'}, inplace=True)


,user_id,country,age,signup_date,subscription_type,subscription_status,months_inactive,is_churned,ad_interaction,ad_conversion_to_subscription,...,favorite_genre,most_liked_feature,desired_future_feature,primary_device,playlists_created,avg_skips_per_day,user_age_days,is_outlier,age_group,engagement_level
0,1,France,25,2021-08-19,Premium Duo,Active,0,0,No,No,...,Bollywood,Radio,Concert Alerts,Tablet,7,8,1706,False,25-34,Medium
1,2,Indonesia,20,2022-06-06,Premium Family,Active,0,0,Yes,No,...,Latin,Podcasts,Lyrics Translation,Mobile,7,6,1415,False,18-24,Medium
2,3,Italy,53,2024-01-04,Premium Individual,Active,0,0,Yes,Yes,...,Bollywood,Lyrics,Better AI Recommendations,Desktop,6,5,838,False,45-54,Low
3,4,Italy,48,2018-08-26,Premium Individual,Active,1,0,No,No,...,Electronic,Playlists,Social Listening,Smart Speaker,11,8,2795,False,45-54,Medium
4,5,Australia,18,2020-05-29,Free,Active,0,0,No,No,...,Indie,Daily Mix,Lyrics Translation,Tablet,10,11,2153,False,<18,Medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49994,49995,France,54,2021-08-23,Premium Individual,Active,1,0,Yes,No,...,Bollywood,Discover Weekly,Lyrics Translation,Desktop,13,18,1702,False,45-54,Medium
49995,49996,India,33,2022-09-23,Free,Active,2,0,Yes,No,...,Latin,Offline Mode,Concert Alerts,Tablet,8,5,1306,False,25-34,Medium
49996,49997,Italy,35,2023-11-17,Premium Family,Active,0,0,No,No,...,Latin,Podcasts,Mood-based Auto Playlists,Car System,11,10,886,False,35-44,Low
49997,49998,Brazil,33,2024-11-14,Premium Individual,Active,0,0,No,No,...,Rock,Lyrics,Lyrics Translation,Smart Speaker,9,13,523,False,25-34,Medium


In [32]:
df['likely_churned'] = np.where((df['months_inactive'] >= 1) & (df['months_inactive'] < 3), 1, 0)

,user_id,country,age,signup_date,subscription_type,subscription_status,months_inactive,is_churned,ad_interaction,ad_conversion_to_subscription,...,most_liked_feature,desired_future_feature,primary_device,playlists_created,avg_skips_per_day,user_age_days,is_outlier,age_group,engagement_level,likely_churned
0,1,France,25,2021-08-19,Premium Duo,Active,0,0,No,No,...,Radio,Concert Alerts,Tablet,7,8,1706,False,25-34,Medium,0
1,2,Indonesia,20,2022-06-06,Premium Family,Active,0,0,Yes,No,...,Podcasts,Lyrics Translation,Mobile,7,6,1415,False,18-24,Medium,0
2,3,Italy,53,2024-01-04,Premium Individual,Active,0,0,Yes,Yes,...,Lyrics,Better AI Recommendations,Desktop,6,5,838,False,45-54,Low,0
3,4,Italy,48,2018-08-26,Premium Individual,Active,1,0,No,No,...,Playlists,Social Listening,Smart Speaker,11,8,2795,False,45-54,Medium,1
4,5,Australia,18,2020-05-29,Free,Active,0,0,No,No,...,Daily Mix,Lyrics Translation,Tablet,10,11,2153,False,<18,Medium,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49994,49995,France,54,2021-08-23,Premium Individual,Active,1,0,Yes,No,...,Discover Weekly,Lyrics Translation,Desktop,13,18,1702,False,45-54,Medium,1
49995,49996,India,33,2022-09-23,Free,Active,2,0,Yes,No,...,Offline Mode,Concert Alerts,Tablet,8,5,1306,False,25-34,Medium,1
49996,49997,Italy,35,2023-11-17,Premium Family,Active,0,0,No,No,...,Podcasts,Mood-based Auto Playlists,Car System,11,10,886,False,35-44,Low,0
49997,49998,Brazil,33,2024-11-14,Premium Individual,Active,0,0,No,No,...,Lyrics,Lyrics Translation,Smart Speaker,9,13,523,False,25-34,Medium,0


Adjust status

In [45]:
df['Final_status'] = df['subscription_status']

df.loc[df['likely_churned'] == 1, 'Final_status'] = 'Likely Churned'

df.loc[df['is_churned'] == 1, 'Final_status'] = 'Churned'

String Cleaning

In [46]:
cols = ['country', 'favorite_genre', 'primary_device']

for col in cols:
    df[col] = df[col].astype(str).str.strip().str.lower()

Tenure Bucket

In [47]:
df['customer_type'] = df['user_age_days'] / 30

conditions = [
    df['customer_type'] < 1,
    (df['customer_type'] >= 1) & (df['customer_type'] < 5),
    df['customer_type'] >= 5
]

choices = ['New', 'Established', 'Loyal']

df['customer_type'] = np.select(conditions, choices, default='Undefined')

Engagement Score
Listening hours = 50% importance
Playlists created = 50% importance

In [83]:
df['engagement_score'] = (
    df['playlists_created'] * 0.3 +
    df['avg_listening_hours_per_week'] * 0.7
)



,user_id,country,age,signup_date,subscription_type,subscription_status,months_inactive,is_churned,ad_interaction,ad_conversion_to_subscription,...,is_outlier,age_group,engagement_level,likely_churned,adjusted_status,tenure_months,tenure_bucket,customer_type,Final_status,engagement_score
0,1,france,25,2021-08-19,Premium Duo,Active,0,0,No,No,...,False,25-34,Medium,0,Active,56.866667,Loyal,Loyal,Active,9.19
1,2,indonesia,20,2022-06-06,Premium Family,Active,0,0,Yes,No,...,False,18-24,Medium,0,Active,47.166667,Loyal,Loyal,Active,10.24
2,3,italy,53,2024-01-04,Premium Individual,Active,0,0,Yes,Yes,...,False,45-54,Medium,0,Active,27.933333,Loyal,Loyal,Active,8.45
3,4,italy,48,2018-08-26,Premium Individual,Active,1,0,No,No,...,False,45-54,High,1,Likely Churned,93.166667,Loyal,Loyal,Likely Churned,12.51
4,5,australia,18,2020-05-29,Free,Active,0,0,No,No,...,False,<18,High,0,Active,71.766667,Loyal,Loyal,Active,11.89
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49994,49995,france,54,2021-08-23,Premium Individual,Active,1,0,Yes,No,...,False,45-54,High,1,Likely Churned,56.733333,Loyal,Loyal,Likely Churned,11.71
49995,49996,india,33,2022-09-23,Free,Active,2,0,Yes,No,...,False,25-34,Medium,1,Likely Churned,43.533333,Loyal,Loyal,Likely Churned,10.68
49996,49997,italy,35,2023-11-17,Premium Family,Active,0,0,No,No,...,False,35-44,Medium,0,Active,29.533333,Loyal,Loyal,Active,8.91
49997,49998,brazil,33,2024-11-14,Premium Individual,Active,0,0,No,No,...,False,25-34,High,0,Active,17.433333,Loyal,Loyal,Active,12.83


Normalize score (optional but recommended)

In [84]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df['engagement_score'] = scaler.fit_transform(df[['engagement_score']])

conditions = [
    df['engagement_score'] < 0.4,
    (df['engagement_score'] >= 0.4) & (df['engagement_score'] < 0.6),
    df['engagement_score'] >= 0.6
]
choices = ['Low', 'Medium', 'High']
df['engagement_level'] = np.select(conditions, choices, default='Unknown')


,user_id,country,age,signup_date,subscription_type,subscription_status,months_inactive,is_churned,ad_interaction,ad_conversion_to_subscription,...,is_outlier,age_group,engagement_level,likely_churned,adjusted_status,tenure_months,tenure_bucket,customer_type,Final_status,engagement_score
0,1,france,25,2021-08-19,Premium Duo,Active,0,0,No,No,...,False,25-34,Medium,0,Active,56.866667,Loyal,Loyal,Active,0.474386
1,2,indonesia,20,2022-06-06,Premium Family,Active,0,0,Yes,No,...,False,18-24,Medium,0,Active,47.166667,Loyal,Loyal,Active,0.530416
2,3,italy,53,2024-01-04,Premium Individual,Active,0,0,Yes,Yes,...,False,45-54,Medium,0,Active,27.933333,Loyal,Loyal,Active,0.434899
3,4,italy,48,2018-08-26,Premium Individual,Active,1,0,No,No,...,False,45-54,High,1,Likely Churned,93.166667,Loyal,Loyal,Likely Churned,0.651547
4,5,australia,18,2020-05-29,Free,Active,0,0,No,No,...,False,<18,High,0,Active,71.766667,Loyal,Loyal,Active,0.618463
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49994,49995,france,54,2021-08-23,Premium Individual,Active,1,0,Yes,No,...,False,45-54,High,1,Likely Churned,56.733333,Loyal,Loyal,Likely Churned,0.608858
49995,49996,india,33,2022-09-23,Free,Active,2,0,Yes,No,...,False,25-34,Medium,1,Likely Churned,43.533333,Loyal,Loyal,Likely Churned,0.553895
49996,49997,italy,35,2023-11-17,Premium Family,Active,0,0,No,No,...,False,35-44,Medium,0,Active,29.533333,Loyal,Loyal,Active,0.459445
49997,49998,brazil,33,2024-11-14,Premium Individual,Active,0,0,No,No,...,False,25-34,High,0,Active,17.433333,Loyal,Loyal,Active,0.668623


Final Validation

In [73]:
df.info()



<class 'pandas.core.frame.DataFrame'>
Index: 49834 entries, 0 to 49999
Data columns (total 29 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   user_id                         49834 non-null  int64         
 1   country                         49834 non-null  object        
 2   age                             49834 non-null  int64         
 3   signup_date                     49834 non-null  datetime64[ns]
 4   subscription_type               49834 non-null  object        
 5   subscription_status             49834 non-null  object        
 6   months_inactive                 49834 non-null  int64         
 7   is_churned                      49834 non-null  int64         
 8   ad_interaction                  49834 non-null  object        
 9   ad_conversion_to_subscription   49834 non-null  object        
 10  music_suggestion_rating_1_to_5  49834 non-null  int64         
 11  avg_lis

In [74]:
df.describe()

df[['user_age_days', 'age_group', 'engagement_level',
    'tenure_bucket', 'engagement_score',
    'adjusted_status']].head()

,user_age_days,age_group,engagement_level,tenure_bucket,engagement_score,adjusted_status
0,1706,25-34,Medium,Loyal,0.474386,Active
1,1415,18-24,Medium,Loyal,0.530416,Active
2,838,45-54,Medium,Loyal,0.434899,Active
3,2795,45-54,High,Loyal,0.651547,Likely Churned
4,2153,<18,High,Loyal,0.618463,Active


In [85]:
df.to_csv('final_cleaned_spotify_data.csv', index=False)


In [86]:
!git add notebooks/02_cleaning.ipynb

# 2. Commit the changes
!git commit -m "Update engagement rounding and churn logic"

# 3. Push to your repository
!git push

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
